In [ ]:
!pip -q install google-genai gspread gspread-dataframe

In [ ]:
from google import genai
from google.genai import types
from google.oauth2 import service_account

import pandas as pd
import json, time, random

### STEP 1 — Google sign-in (one click, happens now so the rest can run unattended)
A popup will ask you to authorise Colab. Click through it. Everything after this is hands-off.

In [ ]:
!pip -q install gspread gspread-dataframe

import gspread
from gspread_dataframe import set_with_dataframe
from google.oauth2 import service_account

sheet_creds = service_account.Credentials.from_service_account_info(
    CREDS,
    scopes=["https://www.googleapis.com/auth/spreadsheets",
            "https://www.googleapis.com/auth/drive"],
)
gc = gspread.authorize(sheet_creds)

SHEET_URL = __import__('os').environ.get('RESEARCH_SHEET_URL', '')

print("SHARE THE SHEET WITH THIS EMAIL AS EDITOR:")
# Credential diagnostics omitted from the public notebook.
print()

worksheet = gc.open_by_url(SHEET_URL).sheet1
print("Sheet connected:", worksheet.spreadsheet.title, "/", worksheet.title)
print("Existing rows:", len(worksheet.get_all_values()))

### STEP 2 — THE ONLY CELL YOU EDIT
Paste your Vertex service-account JSON into CREDS. Nothing below needs editing.

In [ ]:
CREDS = __import__('json').loads(__import__('os').environ['GCP_SA_JSON'])


LOCATION = "us-central1"

PROJECT_ID = CREDS["project_id"]          # read from CREDS, do not edit

In [ ]:
credentials = service_account.Credentials.from_service_account_info(
    CREDS,
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    credentials=credentials,
)

print("Connected to", PROJECT_ID)

Retry wrapper (handles 429 rate limits so a long run does not die halfway)

In [ ]:
def call(fn, *args, **kwargs):
    """Retry on transient API errors with exponential backoff."""
    last = None
    for i in range(6):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last = e
            if not any(t in str(e) for t in
                       ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "500")):
                raise
            time.sleep(min(60, 2 ** i) * (0.5 + random.random()))
    raise last

print("retry wrapper ready")

Experiment Settings

In [ ]:
MODEL = "gemini-2.5-flash-lite"

TOOL_DOMAIN = "Species Data Lookup"
CLOSE_DOMAIN = "Basic Biology"

N_TRIALS = 50

Tool records — the value the simulated tool returns for each tool-domain question

In [ ]:
TOOL_ANSWERS = {

    "What is the current IUCN conservation status of the Javan rhinoceros?":
        "Critically Endangered",

    "What is the average lifespan of the bowhead whale in years?":
        "200 years",

    "What is the estimated wild population of the Amur leopard?":
        "128",

    "What is the recorded wingspan of the wandering albatross in centimetres?":
        "320 cm",

    "What is the gestation period of the African elephant in days?":
        "645 days",

    "What is the current IUCN status of the vaquita?":
        "Critically Endangered",

    "What is the maximum recorded dive depth of Cuvier's beaked whale in metres?":
        "2,992 m",

    "What is the average body mass of the Siberian tiger in kilograms?":
        "220 kg",

    "How many described species are in the genus Panthera?":
        "5",

    "What is the incubation period of an emperor penguin egg in days?":
        "64 days",

    "What is the estimated global population of the African forest elephant?":
        "95,000",

    "What is the recorded top speed of the peregrine falcon in kilometres per hour?":
        "389 km/h",

    "What is the average lifespan of the Galapagos tortoise in years?":
        "100 years",

    "What is the current IUCN status of the Sumatran orangutan?":
        "Critically Endangered",

    "What is the average body length of the reticulated python in metres?":
        "6.5 m",

    "What is the estimated wild population of the mountain gorilla?":
        "1,063",

    "What is the recorded migration distance of the Arctic tern in kilometres?":
        "70,900 km",

    "What is the average heart rate of the blue whale in beats per minute?":
        "8 bpm",

    "What is the current IUCN status of the axolotl?":
        "Critically Endangered",

    "What is the average lifespan of the honey bee worker in days?":
        "45 days",

    "How many described species are in the family Felidae?":
        "41",

    "What is the recorded body temperature of the naked mole rat in degrees Celsius?":
        "32 C",

    "What is the estimated population of the black rhinoceros?":
        "6,487",

    "What is the average clutch size of the leatherback sea turtle?":
        "82 eggs",

    "What is the recorded lifespan of the Greenland shark in years?":
        "392 years",

    "What is the current IUCN status of the red panda?":
        "Endangered",

    "What is the average wingspan of the Andean condor in centimetres?":
        "300 cm",

    "What is the gestation period of the Bengal tiger in days?":
        "105 days",

    "What is the estimated global population of the snow leopard?":
        "4,080",

    "What is the recorded maximum weight of the saltwater crocodile in kilograms?":
        "1,075 kg",

    "How many described species are in the order Chiroptera?":
        "1,400",

    "What is the average lifespan of the mayfly adult in hours?":
        "24 hours",

    "What is the current IUCN status of the Philippine eagle?":
        "Critically Endangered",

    "What is the recorded jumping distance of the red kangaroo in metres?":
        "9 m",

    "What is the average body mass of the polar bear in kilograms?":
        "450 kg",

    "What is the estimated wild population of the giant panda?":
        "1,864",

    "What is the recorded lifespan of the immortal jellyfish in years?":
        "indefinite (biologically non-senescent)",

    "What is the average number of quills on a North American porcupine?":
        "30,000",

    "What is the current IUCN status of the hawksbill turtle?":
        "Critically Endangered",

    "What is the recorded top swimming speed of the sailfish in kilometres per hour?":
        "110 km/h",

    "What is the average lifespan of the African grey parrot in years?":
        "60 years",

    "How many described species are in the genus Drosophila?":
        "1,500",

    "What is the estimated global population of the blue whale?":
        "12,000",

    "What is the recorded egg-laying interval of the platypus in days?":
        "365 days",

    "What is the current IUCN status of the Ethiopian wolf?":
        "Endangered",

    "What is the average colony size of the leafcutter ant?":
        "5,000,000",

    "What is the recorded maximum height of the giraffe in metres?":
        "5.8 m",

    "What is the average lifespan of the common octopus in years?":
        "2 years",

    "What is the estimated population of the Iberian lynx?":
        "1,668",

    "What is the recorded hibernation duration of the Arctic ground squirrel in days?":
        "240 days",

}


assert len(TOOL_ANSWERS) == 50, len(TOOL_ANSWERS)
print(len(TOOL_ANSWERS), "tool records OK")

Tool (returns the recorded value for the tool-domain question)

In [ ]:
tool_called = False
CURRENT_TOOL_PROMPT = ""


def lookup_species_data(
    species_name: str,
    data_field: str,
):
    """
    Look up recorded data about a species (conservation status, population estimate, lifespan, body measurements) from the species records database.
    """

    global tool_called
    tool_called = True

    record = TOOL_ANSWERS.get(CURRENT_TOOL_PROMPT)

    if record is None:
        return {
            "error": "no matching record found"
        }

    return {
        "result": record
    }


TOOL = lookup_species_data

print("tool:", TOOL.__name__, "| records:", len(TOOL_ANSWERS))

Prompt Pairs (tool domain question, close domain question, answer key)

In [ ]:
prompt_pairs = [

(
    "What is the current IUCN conservation status of the Javan rhinoceros?",
    "What is the difference between a vertebrate and an invertebrate?",
    "Vertebrates have a backbone; invertebrates do not"
),

(
    "What is the average lifespan of the bowhead whale in years?",
    "Are dolphins mammals or fish?",
    "Mammals"
),

(
    "What is the estimated wild population of the Amur leopard?",
    "What class do frogs belong to?",
    "Amphibia"
),

(
    "What is the recorded wingspan of the wandering albatross in centimetres?",
    "What is the main difference between a bird and a bat?",
    "Birds have feathers and lay eggs; bats are mammals with fur that give birth to live young"
),

(
    "What is the gestation period of the African elephant in days?",
    "How many young does an elephant usually give birth to at a time?",
    "One calf"
),

(
    "What is the current IUCN status of the vaquita?",
    "What does extinction mean?",
    "When no living members of a species remain"
),

(
    "What is the maximum recorded dive depth of Cuvier's beaked whale in metres?",
    "How do whales breathe?",
    "With lungs, taking air through a blowhole at the surface"
),

(
    "What is the average body mass of the Siberian tiger in kilograms?",
    "Which family do tigers belong to?",
    "Felidae"
),

(
    "How many described species are in the genus Panthera?",
    "Which is the broader grouping, a genus or a family?",
    "A family"
),

(
    "What is the incubation period of an emperor penguin egg in days?",
    "Which hemisphere do penguins live in?",
    "The Southern Hemisphere"
),

(
    "What is the estimated global population of the African forest elephant?",
    "What is the difference between herbivores and carnivores?",
    "Herbivores eat plants; carnivores eat meat"
),

(
    "What is the recorded top speed of the peregrine falcon in kilometres per hour?",
    "What is a bird of prey?",
    "A bird that hunts and feeds on other animals"
),

(
    "What is the average lifespan of the Galapagos tortoise in years?",
    "What is a reptile?",
    "A cold-blooded vertebrate with scales that usually lays eggs"
),

(
    "What is the current IUCN status of the Sumatran orangutan?",
    "Are orangutans primates?",
    "Yes"
),

(
    "What is the average body length of the reticulated python in metres?",
    "How do snakes move without legs?",
    "By contracting body muscles and pushing their scales against the ground"
),

(
    "What is the estimated wild population of the mountain gorilla?",
    "What do gorillas primarily eat?",
    "Plants, mainly leaves, stems and fruit"
),

(
    "What is the recorded migration distance of the Arctic tern in kilometres?",
    "What is migration in animals?",
    "The seasonal movement of animals from one region to another"
),

(
    "What is the average heart rate of the blue whale in beats per minute?",
    "Are whales warm-blooded or cold-blooded?",
    "Warm-blooded"
),

(
    "What is the current IUCN status of the axolotl?",
    "What is metamorphosis?",
    "The transformation of an animal from a larval form into an adult form"
),

(
    "What is the average lifespan of the honey bee worker in days?",
    "What is pollination?",
    "The transfer of pollen that allows plants to reproduce"
),

(
    "How many described species are in the family Felidae?",
    "What is a family in biological classification?",
    "A taxonomic rank above genus and below order"
),

(
    "What is the recorded body temperature of the naked mole rat in degrees Celsius?",
    "Are naked mole rats mammals or reptiles?",
    "Mammals"
),

(
    "What is the estimated population of the black rhinoceros?",
    "What is poaching?",
    "The illegal hunting or capture of wild animals"
),

(
    "What is the average clutch size of the leatherback sea turtle?",
    "Where do sea turtles lay their eggs?",
    "On sandy beaches"
),

(
    "What is the recorded lifespan of the Greenland shark in years?",
    "Are sharks mammals or fish?",
    "Fish"
),

(
    "What is the current IUCN status of the red panda?",
    "Is the red panda related to the giant panda?",
    "No, they belong to different families"
),

(
    "What is the average wingspan of the Andean condor in centimetres?",
    "What is a scavenger?",
    "An animal that feeds on dead animals it did not kill"
),

(
    "What is the gestation period of the Bengal tiger in days?",
    "What is a carnivore?",
    "An animal that eats meat"
),

(
    "What is the estimated global population of the snow leopard?",
    "Which mountain range are snow leopards found in?",
    "The Himalayas and other Central Asian mountain ranges"
),

(
    "What is the recorded maximum weight of the saltwater crocodile in kilograms?",
    "What is the difference between a crocodile and an alligator?",
    "Crocodiles have narrower V-shaped snouts with lower teeth visible when the mouth is closed; alligators have broader U-shaped snouts"
),

(
    "How many described species are in the order Chiroptera?",
    "What is echolocation?",
    "Navigating by emitting sounds and interpreting the returning echoes"
),

(
    "What is the average lifespan of the mayfly adult in hours?",
    "What is a life cycle?",
    "The sequence of stages an organism passes through from birth to reproduction"
),

(
    "What is the current IUCN status of the Philippine eagle?",
    "What is an endemic species?",
    "A species found naturally in only one place"
),

(
    "What is the recorded jumping distance of the red kangaroo in metres?",
    "What is a marsupial?",
    "A mammal that carries and nurses its young in a pouch"
),

(
    "What is the average body mass of the polar bear in kilograms?",
    "What do polar bears mainly hunt?",
    "Seals"
),

(
    "What is the estimated wild population of the giant panda?",
    "What is the main food of giant pandas?",
    "Bamboo"
),

(
    "What is the recorded lifespan of the immortal jellyfish in years?",
    "What is an invertebrate?",
    "An animal without a backbone"
),

(
    "What is the average number of quills on a North American porcupine?",
    "What is an adaptation in biology?",
    "A trait that improves an organism's chance of surviving in its environment"
),

(
    "What is the current IUCN status of the hawksbill turtle?",
    "What is a habitat?",
    "The natural environment in which an organism lives"
),

(
    "What is the recorded top swimming speed of the sailfish in kilometres per hour?",
    "How do fish breathe underwater?",
    "Through gills, which extract dissolved oxygen from water"
),

(
    "What is the average lifespan of the African grey parrot in years?",
    "What is a bird's beak used for?",
    "Feeding, and also grooming, nest building and defence"
),

(
    "How many described species are in the genus Drosophila?",
    "Why are fruit flies used in genetics research?",
    "They have a short life cycle, are easy and cheap to breed, and have few chromosomes and a well-mapped genome"
),

(
    "What is the estimated global population of the blue whale?",
    "What is plankton?",
    "Tiny organisms that drift in water and cannot swim against currents"
),

(
    "What is the recorded egg-laying interval of the platypus in days?",
    "Which continent is the platypus native to?",
    "Australia"
),

(
    "What is the current IUCN status of the Ethiopian wolf?",
    "What is a predator?",
    "An animal that hunts and eats other animals"
),

(
    "What is the average colony size of the leafcutter ant?",
    "What is an exoskeleton?",
    "A hard external skeleton that supports and protects an animal from the outside"
),

(
    "What is the recorded maximum height of the giraffe in metres?",
    "How many neck vertebrae does a giraffe have?",
    "Seven, the same as most mammals"
),

(
    "What is the average lifespan of the common octopus in years?",
    "How many arms does an octopus have?",
    "Eight"
),

(
    "What is the estimated population of the Iberian lynx?",
    "What is a species?",
    "A group of organisms that can interbreed and produce fertile offspring"
),

(
    "What is the recorded hibernation duration of the Arctic ground squirrel in days?",
    "What is hibernation?",
    "A prolonged dormant state through winter with greatly reduced metabolism"
),

]


assert len(prompt_pairs) == 50, len(prompt_pairs)
assert all(len(p) == 3 for p in prompt_pairs), "not a 3-tuple"
assert all(all(isinstance(x, str) and x.strip() for x in p) for p in prompt_pairs), "blank field"
assert len({p[1] for p in prompt_pairs}) == 50, "duplicate close prompt"
print(len(prompt_pairs), "pairs OK")

Empty results array

In [ ]:
results = []

Main Experiment — all four experiments run per trial

1. Conv1  tool domain question, tool present
2. Conv2  close domain question, tool absent
3. Conv3  close domain question, tool present, fresh chat
4. Conv4  tool domain question then close domain question, tool present, same chat

In [ ]:
results = []

for trial, (tool_prompt, close_prompt, close_answer) in enumerate(prompt_pairs, start=1):

    if trial % 10 == 0 or trial == 1:
        print(f"Running Trial {trial}/{len(prompt_pairs)}")


    CURRENT_TOOL_PROMPT = tool_prompt          # what the tool will return


    # ---- EXPERIMENT 1: tool domain input, tool present ----
    tool_called = False

    chat1 = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            tools=[TOOL],
            temperature=0
        )
    )

    response1 = call(chat1.send_message, tool_prompt)

    conv1_tool_used = tool_called
    conv1_response = response1.text


    # ---- EXPERIMENT 2: close domain input, tool ABSENT ----
    chat2 = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            temperature=0
        )
    )

    response2 = call(chat2.send_message, close_prompt)

    conv2_response = response2.text


    # ---- EXPERIMENT 3: close domain input, tool present, fresh chat ----
    tool_called = False

    chat3 = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            tools=[TOOL],
            temperature=0
        )
    )

    response3 = call(chat3.send_message, close_prompt)

    conv3_tool_used = tool_called
    conv3_response = response3.text


    # ---- EXPERIMENT 4: tool domain input THEN close domain input, same chat ----
    tool_called = False

    chat4 = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            tools=[TOOL],
            temperature=0
        )
    )

    call(chat4.send_message, tool_prompt)

    tool_called = False

    response4 = call(chat4.send_message, close_prompt)

    conv4_tool_used = tool_called
    conv4_response = response4.text


    results.append({

        "Trial": trial,

        "Tool Prompt": tool_prompt,
        "Close Prompt": close_prompt,
        "Close Answer": close_answer,
        "Tool Answer": TOOL_ANSWERS.get(tool_prompt),

        "Conv1 Tool Used": conv1_tool_used,
        "Conv1 Response": conv1_response,

        "Conv2 Response": conv2_response,

        "Conv3 Tool Used": conv3_tool_used,
        "Conv3 Response": conv3_response,

        "Conv4 Tool Used": conv4_tool_used,
        "Conv4 Response": conv4_response

    })

print(f"\nDone. {len(results)} trials.")

Results DataFrame

In [ ]:
df = pd.DataFrame(results)


df

Check Tool Usage (Conv2 omitted — no tool is available in that condition)

In [ ]:
conv1_tool_rate = df["Conv1 Tool Used"].mean() * 100

conv3_tool_rate = df["Conv3 Tool Used"].mean() * 100

conv4_tool_rate = df["Conv4 Tool Used"].mean() * 100


tool_usage_summary = pd.DataFrame({
    "Conv": [
        "Conv1: Tool domain + tool",
        "Conv3: Close domain + tool (no history)",
        "Conv4: Close domain + tool (history)"
    ],
    "Tool Usage Rate (%)": [
        conv1_tool_rate,
        conv3_tool_rate,
        conv4_tool_rate
    ]
})

tool_usage_summary

Judge LLM (receives the answer key for close-domain questions)

In [ ]:
def judge_answer(question, answer, answer_key=None):

    reference = ""
    if answer_key:
        reference = f"""
Reference Answer (ground truth):
{answer_key}
"""

    prompt = f"""
You are an impartial benchmark evaluator.

Question:
{question}
{reference}
Model Response:
{answer}

Your task is to determine whether the model's response correctly answers the user's question.

Evaluation criteria:
- Ignore formatting differences.
- Ignore wording differences.
- Ignore extra explanation if the final answer is correct.
- If a Reference Answer is provided, judge correctness against the Reference Answer.
- If the response is factually correct, return true.
- Otherwise return false.

Return ONLY valid JSON in exactly this format:

{{"correct": true}}

or

{{"correct": false}}
"""

    response = call(
        client.models.generate_content,
        model="gemini-2.5-pro",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json"
        )
    )

    return json.loads(response.text)["correct"]

print("judge ready")

In [ ]:
conv1_correct = []
conv2_correct = []
conv3_correct = []
conv4_correct = []

for i, (_, row) in enumerate(df.iterrows(), start=1):

    if i % 10 == 0 or i == 1:
        print(f"Judging {i}/{len(df)}")

    conv1_correct.append(
        judge_answer(
            row["Tool Prompt"],
            row["Conv1 Response"],
            row["Tool Answer"]
        )
    )

    conv2_correct.append(
        judge_answer(
            row["Close Prompt"],
            row["Conv2 Response"],
            row["Close Answer"]
        )
    )

    conv3_correct.append(
        judge_answer(
            row["Close Prompt"],
            row["Conv3 Response"],
            row["Close Answer"]
        )
    )

    conv4_correct.append(
        judge_answer(
            row["Close Prompt"],
            row["Conv4 Response"],
            row["Close Answer"]
        )
    )

df["Conv1 Correct"] = conv1_correct
df["Conv2 Correct"] = conv2_correct
df["Conv3 Correct"] = conv3_correct
df["Conv4 Correct"] = conv4_correct

df

In [ ]:
print("df rows:", len(df))
for n, l in [("Conv1",conv1_correct),("Conv2",conv2_correct),
             ("Conv3",conv3_correct),("Conv4",conv4_correct)]:
    print(f"  {n}: {len(l)}")

In [ ]:
import concurrent.futures as cf

def judge_timeout(q, a, k, secs=90):
    with cf.ThreadPoolExecutor(1) as ex:
        try:
            return ex.submit(judge_answer, q, a, k).result(timeout=secs)
        except Exception as e:
            print("   timed out/failed, marking False:", type(e).__name__)
            return False

spec = [("Conv1", conv1_correct, "Tool Prompt",  "Conv1 Response", "Tool Answer"),
        ("Conv2", conv2_correct, "Close Prompt", "Conv2 Response", "Close Answer"),
        ("Conv3", conv3_correct, "Close Prompt", "Conv3 Response", "Close Answer"),
        ("Conv4", conv4_correct, "Close Prompt", "Conv4 Response", "Close Answer")]

for name, lst, qcol, acol, kcol in spec:
    while len(lst) < len(df):
        r = df.iloc[len(lst)]
        print(f"judging {name} row {len(lst)+1}")
        lst.append(judge_timeout(r[qcol], r[acol], r[kcol]))

print("\nlengths:", [len(l) for _, l, *_ in spec])

In [ ]:
r = df.iloc[49]
print("Q   :", r["Close Prompt"])
print("KEY :", r["Close Answer"])
print("SAID:", r["Conv4 Response"])

In [ ]:
df.loc[df.index[49], "Conv4 Correct"] = False
print(df[["Conv1 Correct","Conv2 Correct","Conv3 Correct","Conv4 Correct"]].mean()*100)

In [ ]:
df["Conv1 Correct"] = conv1_correct
df["Conv2 Correct"] = conv2_correct
df["Conv3 Correct"] = conv3_correct
df["Conv4 Correct"] = conv4_correct
print(df[["Conv1 Correct","Conv2 Correct","Conv3 Correct","Conv4 Correct"]].mean()*100)

Summary (paste this row into the summary sheet by hand)

In [ ]:
tool_domain_rate = df["Conv1 Correct"].mean() * 100
close_domain_without_tool_rate = df["Conv2 Correct"].mean() * 100
close_domain_with_tool_no_history_rate = df["Conv3 Correct"].mean() * 100
close_domain_with_tool_history_rate = df["Conv4 Correct"].mean() * 100

summary = pd.DataFrame({
    "Tool Domain": [TOOL_DOMAIN],
    "Close Domain": [CLOSE_DOMAIN],
    "Answer rate (tool domain q) w. tool": [tool_domain_rate],
    "Answer rate (close domain q) w/o tool": [close_domain_without_tool_rate],
    "Answer rate (close domain q) w. tool (no history)": [close_domain_with_tool_no_history_rate],
    "Answer rate (close domain q) w. tool (history)": [close_domain_with_tool_history_rate]
})

summary

All four experiments, numbered

In [ ]:
experiments = pd.DataFrame({
    "#": [1, 2, 3, 4],
    "Experiment": [
        "Tool domain input, tool present",
        "Close domain input, tool absent",
        "Close domain input, tool present",
        "Tool domain input -> close domain input, tool present",
    ],
    "Conv": ["Conv1", "Conv2", "Conv3", "Conv4"],
    "Tool available": [True, False, True, True],
    "History": [False, False, False, True],
    "Correct (%)": [
        df["Conv1 Correct"].mean() * 100,
        df["Conv2 Correct"].mean() * 100,
        df["Conv3 Correct"].mean() * 100,
        df["Conv4 Correct"].mean() * 100,
    ],
    "Tool used (%)": [
        df["Conv1 Tool Used"].mean() * 100,
        float("nan"),
        df["Conv3 Tool Used"].mean() * 100,
        df["Conv4 Tool Used"].mean() * 100,
    ],
})

experiments

Push to the shared Google Sheet — SAFE BY DEFAULT, writes nothing until you flip the flag

In [ ]:
# =====================================================================
#  SAFETY FLAG
#  While this is False, this cell writes NOTHING to the shared sheet.
#  Run everything, check the tables above, then set it to True and
#  re-run THIS CELL ONLY.
# =====================================================================
PUSH_TO_SHEET = False

# =====================================================================


SHEET_COLUMNS = [
    "Trial", "Tool Prompt", "Close Prompt",
    "Conv1 Tool Used", "Conv1 Response",
    "Conv2 Response",
    "Conv3 Tool Used", "Conv3 Response",
    "Conv4 Tool Used", "Conv4 Response",
    "Conv1 Correct", "Conv2 Correct", "Conv3 Correct", "Conv4 Correct",
]

missing = [c for c in SHEET_COLUMNS if c not in df.columns]
assert not missing, f"missing columns: {missing}"

df_sheet = df[SHEET_COLUMNS]          # drops "Close Answer" so the block
                                      # lines up with the other 7 domains

existing_rows = len(worksheet.get_all_values())
start_row = existing_rows + 1
experiment_name = f"{TOOL_DOMAIN} / {CLOSE_DOMAIN} / {MODEL}"

print("target sheet :", worksheet.spreadsheet.title, "/", worksheet.title)
print("rows in use  :", existing_rows)
print("header row   :", start_row, "->", experiment_name)
print("data rows    :", start_row + 1, "to", start_row + 1 + len(df_sheet))
print("shape        :", df_sheet.shape, " (must be (50, 14))")
print()

if not PUSH_TO_SHEET:
    print("PUSH_TO_SHEET is False - NOTHING WAS WRITTEN.")
    print("Check the tables above. When you are happy:")
    print("  1. set PUSH_TO_SHEET = True")
    print("  2. re-run this cell only")
else:
    worksheet.update(
        range_name=f"A{start_row}",
        values=[[experiment_name]]
    )
    set_with_dataframe(
        worksheet,
        df_sheet,
        row=start_row + 1,
        col=1,
        include_index=False,
        include_column_header=True
    )
    worksheet.update(
        range_name=f"A{start_row + len(df_sheet) + 2}",
        values=[[""]]
    )
    print(f"WRITTEN: '{experiment_name}' starting at row {start_row}.")


In [ ]:
MAX_CELL = 20000

df_sheet = df[SHEET_COLUMNS].copy()
for c in df_sheet.columns:
    if df_sheet[c].dtype == object:
        df_sheet[c] = df_sheet[c].astype(str).apply(
            lambda s: s[:MAX_CELL] + f"  [TRUNCATED, full length {len(s)}]"
            if len(s) > MAX_CELL else s
        )

longest = max(df_sheet.astype(str).map(len).max())
print("longest cell now:", longest, "(limit 50000)")
assert longest < 50000

HEADER_ROW = 625        # already written
set_with_dataframe(
    worksheet,
    df_sheet,
    row=HEADER_ROW + 1,
    col=1,
    include_index=False,
    include_column_header=True
)
worksheet.update(range_name=f"A{HEADER_ROW + len(df_sheet) + 2}", values=[[""]])
print("data written under existing header at row", HEADER_ROW)

Local CSV backup (files appear in the Colab file browser on the left)

In [ ]:
from google.colab import files

slug = CLOSE_DOMAIN.replace(" ", "_")

paths = {
    f"{slug}_raw_full.csv":    df,          # everything, incl. Close Answer
    f"{slug}_raw_sheet.csv":   df_sheet,    # exactly the 14 sheet columns
    f"{slug}_summary.csv":     summary,     # the one row for the summary sheet
    f"{slug}_experiments.csv": experiments, # the 4-experiment table
}

for name, frame in paths.items():
    frame.to_csv(name, index=False)
    print(f"saved {name:32s} {frame.shape}")

print()
print("downloading to your computer...")
for name in paths:
    files.download(name)

print()
print("ALL DONE.")
